# IST Scenario 2 — Did early heparin help, and at what cost in bleeding?

_Weighing benefits against harms in a trial with a three-level exposure. Level: second analysis. About 90 minutes for Part A, one hour for Part B._

**Learning goals.** Compare several outcomes across a treatment with more than two levels;
recognise a dose-response relationship; put a benefit and a harm on the same scale (events per
1,000 patients) and reach a net judgement; explain what a factorial trial is. In Part B: compare
a mean across three groups with a one-way ANOVA; analyse both drugs of a factorial trial in one
logistic model; turn a model into absolute risks for different patients; read a Cox model of
survival and draw mortality curves from federated histograms.

**Clinical background.** In 1991 many physicians gave heparin routinely after an ischaemic
stroke, hoping to stop the thrombus growing and to prevent recurrence and pulmonary embolism.
Others feared haemorrhagic transformation. IST tested two doses of subcutaneous heparin,
5,000 IU twice daily (low) and 12,500 IU twice daily (medium), against no heparin, for 14 days.
Because aspirin was tested at the same time, every patient sits in one of six cells: a 3 x 2
factorial design, in which each drug can be analysed as if the other did not exist.

**Today.** Routine early anticoagulation after ischaemic stroke is not recommended, largely
because of IST. Read this scenario as the origin of that practice, not as a guide to it.

**Before you start.** IST randomised 19,435 patients between 1991 and 1996. Nobody received
thrombolysis or thrombectomy, one third were randomised before a CT scan, and "dependent" was
measured by questionnaire, not by the modified Rankin Scale. The *conclusions* of the trial still
hold and are current practice; the *absolute percentages* describe 1990s stroke care and must not
be quoted to patients.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model).
No patient row ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect.

## 1. Connect to the MIP and open the IST pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("IST")      # add version="1.0" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())        # the pathology label differs: pick it from this list
    raise
ist = dm.datasets.list()[0]             # the IST pathology holds a single dataset
display(dm)
display(ist)

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    out = {}
    for item in raw:
        if isinstance(item, dict):
            out[str(item.get("code"))] = str(item.get("label"))
        else:
            out[str(item)] = str(item)
    return out


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the IST dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ist], variables=variables), filters=filters,
                        handle_missing=handler)


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == "all datasets"]


def numeric_summary(result, variables):
    """Mean, SD, median and range of numeric variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "median": d.get("q2"),
                         "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def describe_by(variables, group, base_filter=None):
    """Describe the variables separately in each level of `group` (one federated call per level).
    Returns (numeric table, categorical table) with a column per group level."""
    num, cat = [], []
    for c, label in labels_of(group).items():
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(variables, f).describe()
        n = numeric_summary(res, [v for v in variables if v.is_numerical()])
        if len(n):
            num.append(n.assign(group=label))
        for v in variables:
            if v.is_categorical():
                k = category_counts(res, v)
                if len(k):
                    cat.append(k.assign(group=label))
    num = pd.concat(num, ignore_index=True) if num else pd.DataFrame()
    cat = pd.concat(cat, ignore_index=True) if cat else pd.DataFrame()
    if len(num):
        num = num.pivot(index="variable", columns="group", values=["n", "mean", "sd"])
    if len(cat):
        cat = cat.pivot(index=["variable", "level"], columns="group", values="percent")
    return num, cat


def crosstab(exposure, outcome, base_filter=None, event="1"):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    rows = []
    for c, label in lab_x.items():
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: label, "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row["% " + lab_y.get(event, event)] = 100 * counts.get(event, 0) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: "{:.1f}"}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.2f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(5, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first level of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The variables of this scenario

| Code | Label in the MIP | Tree location (under IST) | Role | Values |
|---|---|---|---|---|
| `RXHEP` | Heparin allocated | Acute treatment / Trial allocation | exposure, 3 levels | 0 Avoid heparin; 1 Low dose: 5000 IU subcutaneously twice daily; 2 Medium dose: 12500 IU subcutaneously twice daily |
| `RXHEPANY` | Any heparin allocated | Acute treatment / Trial allocation | exposure, 2 levels; curation-derived | 0 No; 1 Yes |
| `TRTGRP` | Treatment group | Acute treatment / Trial allocation | the six cells of the factorial design; curation-derived | 1 Aspirin + medium-dose heparin; 2 Medium-dose heparin alone; 3 Aspirin + low-dose heparin; 4 Low-dose heparin alone; 5 Aspirin alone; 6 Neither (avoid aspirin, avoid heparin) |
| `RXASP` | Aspirin allocated | Acute treatment / Trial allocation | the other factor of the design | 0 No; 1 Yes |
| `DEADDEP6` | Dead or dependent at 6 months | Follow-up at 6 months / Functional outcome | primary outcome; curation-derived | 0 No; 1 Yes |
| `ID14` | Died within 14 days | Death and survival / Survival analysis | early outcome | 0 No; 1 Yes |
| `ISC14` | Ischaemic stroke within 14 days | Complications within 14 days / Main event indicators | expected benefit | 0 No; 1 Yes |
| `PE14` | Pulmonary embolism within 14 days | Complications within 14 days / Main event indicators | expected benefit | 0 No; 1 Yes |
| `H14` | Cerebral bleed within 14 days | Complications within 14 days / Main event indicators | expected harm | 0 No; 1 Yes |
| `TRAN14` | Major non-cerebral bleed within 14 days | Complications within 14 days / Main event indicators | expected harm | 0 No; 1 Yes |
| `AGE` | Age | Demographics and event / Demographics | Part B: balance check, covariate | integer, 16-99 years |
| `RSBP` | Systolic blood pressure | Clinical assessment at admission / Consciousness and vitals | Part B: balance check | integer, 50-300 mmHg |
| `EXPDD` | Predicted death or dependence at 6 months | Predicted risk (IST prognostic model) | Part B: balance check (prognosis at baseline) | real, 0-1 |
| `RCONSC` | Conscious level | Clinical assessment at admission / Consciousness and vitals | Part B: covariate | 1 Fully alert; 2 Drowsy; 3 Unconscious |
| `RATRIAL` | Atrial fibrillation | Prestroke characteristics / Cerebrovascular risk factors | Part B: covariate of the bleeding models; empty in the pilot phase (984 patients) | 0 No; 1 Yes |
| `TD` | Time to death or censoring | Death and survival / Survival analysis | Part B: follow-up time of the survival analysis | integer, 0-1400 days |
| `DIED` | Died | Death and survival / Survival analysis | Part B: event of the survival analysis | 0 No; 1 Yes |

In [ ]:
rxhep    = dm.variables["Heparin allocated"]
rxhepany = dm.variables["Any heparin allocated"]
trtgrp   = dm.variables["Treatment group"]
rxasp    = dm.variables["Aspirin allocated"]
dd6      = dm.variables["Dead or dependent at 6 months"]
id14     = dm.variables["Died within 14 days"]
isc14    = dm.variables["Ischaemic stroke within 14 days"]
pe14     = dm.variables["Pulmonary embolism within 14 days"]
h14      = dm.variables["Cerebral bleed within 14 days"]
tran14   = dm.variables["Major non-cerebral bleed within 14 days"]

display(mip.to_frame([rxhep, rxhepany, trtgrp, rxasp, dd6, id14, isc14, pe14, h14, tran14]))

# Part A. Benefits and harms, in percentages

## Step 1. See the factorial design

*Expected:* the four cells that include heparin hold about 2,430 patients each, the two cells
without heparin about 4,860 each. The grid below draws the six cells as the 3 x 2 design.
Crossing aspirin against heparin shows that every heparin group has exactly half of its
patients on aspirin: this balance is what allows each drug to be analysed separately.

In [ ]:
design = analysis([trtgrp]).describe()
cells = category_counts(design, trtgrp)
display(cells.style.format({"percent": "{:.1f}"}).hide(axis="index"))

# the same counts as the 3 x 2 grid of the design. Treatment group codes: 1 aspirin + medium dose,
# 2 medium dose alone, 3 aspirin + low dose, 4 low dose alone, 5 aspirin alone, 6 neither
counts = rows_for(design, trtgrp)[0]["data"]["counts"]
grid = pd.DataFrame({"No aspirin": [counts.get(c, 0) for c in ("6", "4", "2")],
                     "Aspirin": [counts.get(c, 0) for c in ("5", "3", "1")]},
                    index=["Avoid heparin", "Low dose", "Medium dose"])
fig, ax = plt.subplots(figsize=(5, 3))
ax.imshow(grid.values, cmap="Blues", vmin=0)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        n = int(grid.iat[i, j])
        ax.text(j, i, f"{n:,}", ha="center", va="center", color="white" if n > 0.7 * grid.values.max() else "black")
ax.set_xticks(range(grid.shape[1]))
ax.set_xticklabels(grid.columns)
ax.set_yticks(range(grid.shape[0]))
ax.set_yticklabels(grid.index)
ax.set_title("Patients in each cell of the 3 x 2 factorial design")
plt.tight_layout()
plt.show()

# aspirin within each heparin group: the percentages should all be 50%
show_crosstab(crosstab(rxhep, rxasp), "Aspirin allocation within each heparin group")

## Step 2. Primary outcome, heparin versus no heparin

*Expected:* 6,063 of 9,641 (62.9%) dead or dependent with heparin versus 6,062 of 9,644 (62.9%)
without. Identical, p = 0.98. This is the Lancet's result to the decimal.

In [ ]:
show_crosstab(crosstab(rxhepany, dd6), "Dead or dependent at 6 months, any heparin vs none")

## Steps 3 and 4. Primary outcome and early death, by dose

*Expected:* dead or dependent 62.9% (avoid), 63.1% (low), 62.6% (medium), p = 0.88. Died within
14 days 9.3%, 8.7%, 9.3%, p = 0.47. No dose does better on either outcome. The last chart adds
the 95% confidence interval of each percentage: the intervals of the three doses overlap almost
completely.

In [ ]:
dd6_by_dose = crosstab(rxhep, dd6)
id14_by_dose = crosstab(rxhep, id14)
show_crosstab(dd6_by_dose, "Dead or dependent at 6 months, by heparin dose")
show_crosstab(id14_by_dose, "Died within 14 days, by heparin dose")

# the same percentages with their 95% confidence intervals: p plus or minus 1.96 standard errors
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, t, title in zip(axes, (dd6_by_dose, id14_by_dose), ("Dead or dependent at 6 months", "Died within 14 days")):
    p = t["Yes"] / t["n"]
    half_width = 1.96 * np.sqrt(p * (1 - p) / t["n"])
    ax.errorbar(range(len(t)), 100 * p, yerr=100 * half_width, fmt="o", capsize=6)
    ax.set_xticks(range(len(t)))
    ax.set_xticklabels([s.split(":")[0] for s in t["Heparin allocated"]])
    ax.set_ylabel("% with 95% CI")
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Steps 5 and 6. The expected benefits and the harms, by dose

*Expected:*

| Event within 14 days | Avoid (n 9,718) | Low dose (n 4,861) | Medium dose (n 4,856) | p |
|---|---|---|---|---|
| Recurrent ischaemic stroke | 223 (2.3%) | 71 (1.5%) | 101 (2.1%) | 0.003 |
| Pulmonary embolism | 81 (0.8%) | 33 (0.7%) | 20 (0.4%) | 0.015 |
| Intracranial haemorrhage | 41 (0.4%) | 35 (0.7%) | 85 (1.8%) | < 0.001 |
| Major extracranial bleed | 37 (0.4%) | 30 (0.6%) | 99 (2.0%) | < 0.001 |

Heparin did prevent some recurrent strokes and pulmonary emboli. Both harms rise with dose: a
monotonic increase is a dose-response relationship, one of the strongest arguments that the drug
causes the event.

In [ ]:
events = {"Recurrent ischaemic stroke": isc14, "Pulmonary embolism": pe14,
          "Intracranial haemorrhage": h14, "Major extracranial bleed": tran14}
tables = {name: crosstab(rxhep, v) for name, v in events.items()}

rate = pd.DataFrame({name: t.set_index("Heparin allocated")["% Yes"] * 10 for name, t in tables.items()}).T
rate.columns = [c.split(":")[0] for c in rate.columns]          # short dose labels
rate["p"] = [tables[name].attrs["p"] for name in rate.index]
rate.index.name = "events per 1,000 patients"
display(rate.style.format({c: "{:.0f}" for c in rate.columns[:-1]} | {"p": fmt_p}))

ax = rate.drop(columns="p").plot(kind="bar", figsize=(8, 4), color=["#8C8C8C", "#4C72B0", "#C44E52"])
ax.set_ylabel("events per 1,000 patients")
ax.set_title("14-day events by heparin dose: benefits (left) and harms (right)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## Step 7. Put benefit and harm on one scale

For the medium dose versus no heparin, per 1,000 patients: events prevented (ischaemic stroke,
pulmonary embolism) against events caused (intracranial haemorrhage, major bleed). The code
below computes the balance sheet from the tables of Step 5 and 6.

*Expected:* roughly 2 ischaemic strokes and 4 pulmonary emboli prevented against 13 intracranial
haemorrhages and 16 major bleeds caused. The harms outnumber the benefits, and an intracranial
haemorrhage is at least as disabling as an ischaemic stroke. That is why the net effect on death
or dependency is zero.

In [ ]:
avoid, low, medium = rate.columns[:3]
balance = pd.DataFrame({
    "medium dose vs avoid, per 1,000": rate[medium] - rate[avoid],
    "low dose vs avoid, per 1,000": rate[low] - rate[avoid],
})
balance["reading"] = np.where(balance.iloc[:, 0] < 0, "prevented", "caused")
display(balance.style.format({balance.columns[0]: "{:+.1f}", balance.columns[1]: "{:+.1f}"}))

ax = balance.iloc[:, 0].plot(kind="barh", figsize=(7, 3), color=np.where(balance.iloc[:, 0] < 0, "#55A868", "#C44E52"))
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("extra events per 1,000 patients (negative = prevented)")
ax.set_ylabel("")
ax.set_title("Balance sheet of medium-dose heparin")
plt.tight_layout()
plt.show()

## Step 8. State the result

> In IST, subcutaneous heparin started within 48 hours of ischaemic stroke had no effect on death
> or dependency at six months (62.9% in both groups) or on 14-day mortality. It prevented a few
> early recurrent ischaemic strokes and pulmonary emboli, but caused a dose-dependent excess of
> intracranial haemorrhage (0.4%, 0.7%, 1.8% for no, low and medium dose) and of major
> extracranial bleeding that cancelled the benefit. This is the origin of the current
> recommendation against routine early anticoagulation in acute ischaemic stroke.

**Questions for discussion**

1. Steps 2 to 4 show "no effect". Is absence of a significant difference the same as evidence of
   no difference? What does the size of the trial allow you to say about how large a benefit
   could still be hidden?
2. Heparin prevented pulmonary embolism. Why is that not enough to recommend it? How would you
   prevent pulmonary embolism in these patients today?
3. Aspirin (Scenario 1) also increased extracranial bleeding, yet its net effect was positive.
   Compare the two balance sheets.

# Part B. The same trial seen through models

Part A answered the trial's questions with percentages. Part B asks the same data four new
questions with the modelling algorithms of the MIP, each of which gives a picture as well as a
table: were the three arms balanced (Step 9), what do both drugs do in one model (Step 10), which
patients bleed (Step 11), and did heparin change how fast patients died (Step 12)? Each step names
the algorithm and the variables to choose in the MIP interface; the notebook runs the same
algorithms and draws the charts. The cohort is every randomised patient unless a step says
otherwise. Run the next two cells first.

In [ ]:
# ---- Toolbox, part 2: helpers for the models of Part B. Run this cell once. ----
def anova_summary(result):
    """One-way ANOVA result -> F, p, a table of the group means and SDs, and the Tukey comparisons."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    table, ci = raw.get("anova_table", {}), raw.get("ci_info", {})
    groups = pd.DataFrame({"mean": pd.Series(ci.get("means", {})), "sd": pd.Series(ci.get("sample_stds", {}))})
    return table.get("f_stat"), table.get("p_value"), groups, pd.DataFrame(raw.get("tuckey_test", []))


def term_label(term, variables):
    """Model term -> readable label: 'RXHEP[2]' -> 'Heparin allocated = Medium dose', 'AGE' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level).split(':')[0]}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def hazard_ratio_table(result, variables):
    """Cox regression result -> one row per term with hazard ratio, 95% CI and p."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    s = raw.get("summary", {})
    terms = raw.get("indep_vars") or s.get("indep_vars") or []
    return pd.DataFrame({"term": [term_label(t, variables) for t in terms],
                         "hazard_ratio": s.get("hazard_ratios"), "ci_low": s.get("hr_lower_ci"),
                         "ci_high": s.get("hr_upper_ci"), "p": s.get("pvalues")})


def forest(ax, table, ratio, title):
    """Forest plot into ax: one ratio (column odds_ratio or hazard_ratio) per term with its 95% CI."""
    t = table.iloc[::-1].reset_index(drop=True)
    ax.errorbar(t[ratio], t.index, xerr=[t[ratio] - t["ci_low"], t["ci_high"] - t[ratio]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ticks = [x for x in (0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 100) if t["ci_low"].min() <= x <= t["ci_high"].max()]
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{x:g}" for x in ticks])
    ax.minorticks_off()
    ax.set_yticks(t.index)
    ax.set_yticklabels(t["term"])
    ax.set_xlabel(ratio.replace("_", " ") + " (log scale)")
    ax.set_title(title, fontsize=10)


print("Toolbox part 2 ready.")

In [ ]:
age     = dm.variables["Age"]
rconsc  = dm.variables["Conscious level"]
ratrial = dm.variables["Atrial fibrillation"]
rsbp    = dm.variables["Systolic blood pressure"]
expdd   = dm.variables["Predicted death or dependence at 6 months"]
td      = dm.variables["Time to death or censoring"]
died    = dm.variables["Died"]
dose_name = {c: label.split(":")[0] for c, label in labels_of(rxhep).items()}   # "2" -> "Medium dose"

display(mip.to_frame([age, rconsc, ratrial, rsbp, expdd, td, died]))

## Step 9. Were the three arms balanced? One-way ANOVA

*One-way ANOVA* compares the mean of a numerical variable across several groups: it extends the
t-test to more than two groups. The F statistic asks whether the group means differ more than
chance alone would make them differ; Tukey's comparisons then test each pair of groups. In the MIP
interface: *One-way ANOVA* with outcome `Age` and predictor `Heparin allocated`, then the same with
`Systolic blood pressure`, then with `Predicted death or dependence at 6 months` as the outcome,
the IST model's prediction for each patient from the admission data: a one-number summary of the
prognosis at baseline.

*Expected:*

| Outcome | Avoid heparin | Low dose | Medium dose | F | p |
|---|---|---|---|---|---|
| Age, mean (years) | 71.7 | 71.7 | 71.8 | 0.27 | 0.76 |
| Systolic blood pressure, mean (mmHg) | 159.9 | 161.0 | 159.8 | 3.41 | 0.033 |
| Predicted death or dependence, mean | 0.629 | 0.629 | 0.629 | 0.02 | 0.98 |

Age and baseline prognosis are identical in the three arms. Systolic pressure differs
"significantly": the low-dose arm is 1.1 mmHg higher than the avoid arm (Tukey p = 0.049) and 1.3
mmHg higher than the medium-dose arm (p = 0.057). In a randomised trial a baseline difference is
due to chance by construction: test enough baseline variables and some p-values fall below 0.05.
And 1 mmHg is clinically meaningless. This is why the CONSORT guideline for reporting trials
advises against significance tests of baseline differences. The charts show each mean with one
standard deviation on either side: the spread between patients dwarfs the differences between
the arms. The platform reports Tukey p-values between 0.001 and 0.9, so 0.9 reads "0.9 or more".

In [ ]:
rows, tukey = [], {}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, v in zip(axes, (age, rsbp, expdd)):
    res = analysis([v, rxhep], drop_missing=True).anova_oneway(group_by=rxhep, outcome=v)
    f_stat, p, groups, pairs = anova_summary(res)
    tukey[v.label] = pairs
    names = [dose_name.get(c, c) for c in groups.index]
    rows.append({"outcome": v.label, **dict(zip(names, groups["mean"])), "F": f_stat, "p": p})
    ax.errorbar(range(len(groups)), groups["mean"], yerr=groups["sd"], fmt="o", capsize=6)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(names)
    ax.set_title(v.label, fontsize=10)
axes[0].set_ylabel("mean, plus or minus 1 SD")
plt.tight_layout()
plt.show()

display(pd.DataFrame(rows).style.format({"F": "{:.2f}", "p": fmt_p} | {n: "{:.3f}" for n in names}).hide(axis="index"))

# Tukey comparisons for systolic blood pressure; diff = mean of group A minus mean of group B
pairs = tukey[rsbp.label].copy()
for col in ("groupA", "groupB"):
    pairs[col] = pairs[col].astype(str).map(dose_name)
display(pairs.style.format({c: "{:.3f}" for c in ("meanA", "meanB", "diff", "se", "t_stat", "p_tuckey")}).hide(axis="index"))

## Step 10. The factorial trial in one model: Logistic Regression

One model can estimate both drugs of the factorial design at once, each adjusted for the other
and for the two strongest prognostic factors. In the MIP interface: *Logistic Regression* with
outcome `Dead or dependent at 6 months` (positive class Yes, code 1) and predictors `Heparin
allocated`, `Aspirin allocated`, `Conscious level` and `Age`. Each categorical predictor is
compared with its first code: avoid heparin, no aspirin, fully alert.

*Expected (19,285 patients), odds ratios:* heparin low dose 1.01 (95% CI 0.94 to 1.10) and medium
dose 0.98 (0.90 to 1.06); aspirin 0.94 (0.88 to 1.00, p = 0.050); drowsy 5.8 and unconscious 33
times the odds of fully alert patients; age 1.06 per year. The forest plot is the whole trial in
one picture: both heparin intervals straddle 1, the aspirin interval just reaches it.

In [ ]:
covariates = [rxhep, rxasp, rconsc, age]
trial_model = analysis([dd6] + covariates, F(dd6).is_not_null(), drop_missing=True).logistic_regression(
    x=covariates, y=dd6, positive_class="1")
trial_ors = odds_ratio_table(trial_model, covariates)
trial_ors["term"] = trial_ors["term"].str.split(":").str[0]          # short dose labels
show_odds_ratios(trial_ors, "Death or dependency at 6 months: both drugs in one model")

## Step 11. Who bleeds? Two logistic regressions, then absolute risks

Both bleeds rose with the dose of heparin (Steps 5 and 6). Here each bleed gets its own model,
with the same covariates, to see which patients carry the risk. In the MIP interface: *Logistic
Regression* with outcome `Cerebral bleed within 14 days`, then `Major non-cerebral bleed within
14 days` (positive class Yes, code 1), and predictors `Heparin allocated`, `Aspirin allocated`,
`Atrial fibrillation`, `Conscious level` and `Age`. The 984 pilot-phase patients without atrial
fibrillation data drop out.

*Expected (18,451 patients; 148 cerebral and 162 major non-cerebral bleeds), odds ratios with 95% CI:*

| Term | Cerebral bleed | Major non-cerebral bleed |
|---|---|---|
| Heparin low dose (vs avoid) | 1.86 (1.15 to 2.98) | 1.63 (1.00 to 2.64) |
| Heparin medium dose (vs avoid) | 4.50 (3.03 to 6.69) | 5.24 (3.57 to 7.67) |
| Aspirin | 1.15 (0.83 to 1.60) | 1.86 (1.34 to 2.57) |
| Atrial fibrillation | 1.82 (1.23 to 2.69) | 0.81 (0.53 to 1.22) |
| Drowsy (vs fully alert) | 2.46 (1.75 to 3.46) | 1.21 (0.84 to 1.73) |
| Unconscious (vs fully alert) | 2.68 (0.97 to 7.45) | 2.26 (0.91 to 5.65) |
| Age, per year | 0.974 (0.962 to 0.987) | 1.045 (1.028 to 1.062) |

Both bleeds rise steeply with the dose. Everything else differs. A cerebral bleed follows the
brain: atrial fibrillation, whose emboli cause large infarcts prone to haemorrhagic
transformation, and reduced consciousness, a marker of a large stroke. A major extracranial bleed
follows the drugs and the patient: aspirin nearly doubles its odds, and each year of age adds
about 4.5%.

In [ ]:
covariates = [rxhep, rxasp, ratrial, rconsc, age]
bleeds = {}
for y in (h14, tran14):
    model = analysis([y] + covariates, F(y).is_not_null(), drop_missing=True).logistic_regression(
        x=covariates, y=y, positive_class="1")
    table = odds_ratio_table(model, covariates)
    table["term"] = table["term"].str.split(":").str[0]
    bleeds[y.label] = (model, table)
    display(Markdown(f"**{y.label}**"))
    display(table.style.format({"odds_ratio": "{:.2f}", "ci_low": "{:.2f}", "ci_high": "{:.2f}", "p": fmt_p}).hide(axis="index"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (label, (model, table)) in zip(axes, bleeds.items()):
    forest(ax, table, "odds_ratio", label)
plt.tight_layout()
plt.show()

### From odds ratios to absolute risks

An odds ratio is relative: the same 4.5 can mean few or many extra bleeds, depending on the
patient's risk without heparin. The model gives a risk for any patient: add the intercept, 72
times the age coefficient and the coefficients of the patient's characteristics to obtain z; the
risk is 1 / (1 + exp(-z)). The cell below prints the coefficients and does this for three patients
aged 72 without aspirin. In the MIP interface, compute one of them by hand from the coefficient
table.

*Expected risk of a cerebral bleed within 14 days:*

| Patient aged 72, no aspirin | Avoid heparin | Low dose | Medium dose |
|---|---|---|---|
| Fully alert, no atrial fibrillation | 0.2% | 0.4% | 1.0% |
| Fully alert, atrial fibrillation | 0.4% | 0.8% | 1.8% |
| Drowsy, atrial fibrillation | 1.0% | 1.9% | 4.4% |

The medium dose adds about 8 cerebral bleeds per 1,000 patients of the lowest-risk profile, but
34 per 1,000 drowsy patients in atrial fibrillation: the patients in whom anticoagulation was most
tempting carried the highest risk.

In [ ]:
coef = bleeds[h14.label][0].to_frame().set_index("feature")["coefficient"]
print(coef.round(4).to_string())

profiles = {"Fully alert, no AF": [],
            "Fully alert, AF": ["RATRIAL[1]"],
            "Drowsy, AF": ["RATRIAL[1]", "RCONSC[2]"]}
risk = pd.DataFrame(index=list(profiles), columns=list(dose_name.values()), dtype=float)
for name, terms in profiles.items():
    for level, dose in dose_name.items():
        z = coef["Intercept"] + coef["AGE"] * 72 + sum(coef[t] for t in terms)
        z += coef.get(f"RXHEP[{level}]", 0.0)          # the reference dose (avoid heparin) has no term
        risk.loc[name, dose] = 100 / (1 + np.exp(-z))
display(risk.style.format("{:.1f}%"))

ax = risk.T.plot(marker="o", figsize=(7, 3.5))
ax.set_ylabel("risk within 14 days (%)")
ax.set_title("Cerebral bleed: absolute risk by heparin dose, age 72, no aspirin")
plt.tight_layout()
plt.show()

## Step 12. Death over time: Cox regression and mortality curves

Steps 4 and 8 looked at death at fixed times. A Cox model uses the day of every death and the
follow-up of every patient, and estimates hazard ratios: how much faster patients in a group die,
at any moment, than patients in the reference group. In the MIP interface: filter `Time to death
or censoring > 0`; outcome `Time to death or censoring`; predictors `Died`, `Heparin allocated`,
`Aspirin allocated`, `Conscious level` and `Age`; then *Cox Proportional Hazards Regression* with
event variable `Died` and event of interest Yes (code 1). The filter is needed because the algorithm
requires strictly positive times: 55 patients died on the day of randomisation and have a time
of 0 days. Two more patients have no recorded time.

*Expected (19,378 patients, 4,315 deaths), hazard ratios with 95% CI:* heparin low dose 1.03 (0.96
to 1.11), medium dose 1.08 (1.01 to 1.16, p = 0.035); aspirin 0.95 (0.89 to 1.01); drowsy 3.4 and
unconscious 9.4 versus fully alert; age 1.05 per year. Without the adjustment for consciousness
and age, the medium-dose hazard ratio is 1.06 (0.99 to 1.14, p = 0.11).

In [ ]:
survival_covariates = [rxhep, rxasp, rconsc, age]
cox = analysis([td, died] + survival_covariates, F(td) > 0, drop_missing=True).cox_regression_classical(
    time=td, event_var=died, covariates=survival_covariates, positive_class="1")
print({k: cox.raw["summary"].get(k) for k in ("n_obs", "n_events")})
hrs = hazard_ratio_table(cox, survival_covariates)
display(hrs.style.format({"hazard_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 3.5))
forest(ax, hrs, "hazard_ratio", "Hazard of death, adjusted (Cox model)")
plt.tight_layout()
plt.show()

### When did the deaths happen? Mortality curves from histograms

The MIP does not draw survival curves, but a histogram of the day of death shows when deaths
occurred. The cell below asks for one histogram per arm, 13 bins of `Time to death or censoring`
with the filters `Died = 1` and `Time to death or censoring` at most 182 days. Each bar holds the
deaths of about two weeks. Adding up the bars from the left and dividing by the size of the arm
(Steps 5 and 6) gives the share of the arm dead by the end of each bin. No bar holds fewer than
10 deaths, the platform's minimum, so no count is hidden. In the MIP interface, the preview of
the same filters draws the histogram with bars of its own width, and the patient counter of the
filter editor gives points of the curve: the deaths at most 14, 84 and 182 days after entry in
each arm (protocol, Step 12).

*Expected:* dead by the end of the first bin (two weeks) 9.0%, 8.3% and 9.1% (avoid, low, medium
dose); by the end of the sixth (twelve weeks) 17.5%, 17.6% and 18.8%; at six months 21.3%, 21.8%
and 22.7%. The bins stop just before day 14 and day 84: the counter of the interface, which
includes those days, gives 9.3%, 8.7% and 9.3% at 14 days and 17.6%, 17.6% and 18.8% at 84 days.
The curves overlap during the first month, when most deaths occur; then the medium-dose curve
drifts above the others. This is what the hazard ratio of 1.08 summarises. A few patients were
followed for less than six months, so the last weeks are slightly underestimated; the Cox model
takes their shorter follow-up into account.

In [ ]:
# patients per arm, from a describe of the allocation
arm_size = rows_for(analysis([rxhep]).describe(), rxhep)[0]["data"]["counts"]

fig, ax = plt.subplots(figsize=(7, 4))
for level, dose in dose_name.items():
    deaths = analysis([td], (F(died) == "1") & (F(td) <= 182) & (F(rxhep) == level),
                      drop_missing=True).histogram(variable=td, bins=13)
    hist = deaths.raw["histogram"][0]
    counts = [c or 0 for c in hist["counts"]]           # a bar under 10 deaths would come back empty (None)
    dead_pct = 100 * np.cumsum(counts) / arm_size[level]
    edges = hist["bins"]                                 # bin edges: one more than the counts
    ax.plot(edges, [0] + list(dead_pct), marker="o", markersize=3, label=dose)
ax.set_xlabel("days since randomisation")
ax.set_ylabel("% of the arm dead")
ax.set_title("Cumulative mortality over six months, by heparin dose")
ax.legend()
plt.tight_layout()
plt.show()

## Step 13. State what the models add

> One-way ANOVA confirmed that randomisation balanced the three heparin arms on age and on
> predicted prognosis; the one "significant" baseline difference, 1 mmHg of systolic pressure, is
> chance. Analysed in one logistic model, the factorial trial showed no effect of either heparin
> dose on death or dependency (odds ratios 1.01 and 0.98) and a small benefit of aspirin (0.94).
> Heparin multiplied the odds of a cerebral bleed by 1.9 at low and 4.5 at medium dose; atrial
> fibrillation and reduced consciousness added to that risk, while aspirin and age added to the
> risk of a major extracranial bleed. In absolute terms, the medium dose caused about 8 extra
> cerebral bleeds per 1,000 low-risk patients and 34 per 1,000 drowsy patients in atrial
> fibrillation. A Cox model found no survival benefit and a possible small excess of deaths with
> the medium dose (hazard ratio 1.08, 95% CI 1.01 to 1.16), a secondary finding.

**Questions for discussion**

4. In Step 9 one baseline variable differs "significantly". Should the analyses of Part A be
   adjusted for it? What would you write in a trial protocol before seeing any data?
5. Older age appears to protect against cerebral bleeds (Step 11). How was a cerebral bleed
   detected in 1991 to 1996, and which patients were most likely to have a repeat CT scan? What
   other explanations are there?
6. The Cox model finds a small excess of deaths with the medium dose that death or dependency at
   six months did not show. Which analysis was pre-specified? With twenty secondary analyses and
   no true effect, how many p-values below 0.05 would you expect by chance?

## Optional extension: does the harm of heparin depend on aspirin?

Cross-tabulate the six treatment cells against intracranial haemorrhage and compare medium-dose
heparin with and without aspirin. Then model the bleed on both factors with *Logistic
Regression*: this is how trialists check whether the two factors of a factorial trial interact
(here, only additively; the platform does not fit interaction terms).

In [ ]:
show_crosstab(crosstab(trtgrp, h14), "Cerebral bleed within 14 days, by treatment cell")

covariates = [rxhep, rxasp]
bleed_model = analysis([h14] + covariates, F(h14).is_not_null(), drop_missing=True).logistic_regression(
    x=covariates, y=h14, positive_class="1")
show_odds_ratios(odds_ratio_table(bleed_model, covariates), "Odds of cerebral bleed within 14 days")